# IT Term Extraction — Training Notebook

This notebook builds a system that scans OJT (On-the-Job Training) documents submitted by students and
identifies IT-related terms (languages, frameworks, tools, databases, cloud platforms, concepts, etc.).

**Approach — three layers, and why:**

1. **Rule-based matching (PhraseMatcher / EntityRuler)** against a predefined CSV list of terms.
   This guarantees every term you already know about (`data/it_terms.csv`) is found and counted exactly,
   with no training required. This alone can already power the deployment script.
2. **Context-pattern mining** (`scripts/context_miner.py`, dependency-parse based) for terms that are
   **not** in the CSV — it looks for phrasing like "I integrated **Stripe API**" or "worked with
   **Terraform**" and pulls out the object regardless of whether it's on your list. Needs no
   training at all.
3. **Trainable statistical NER model** (`spaCy`'s NER component), *bootstrapped* from layers 1 and 2.
   Training on both the exact-CSV labels and the generic context-pattern labels teaches the model
   the underlying *pattern*, so it generalizes to terms it has never seen by name — not just the
   ones already in your CSV.

**Requires** `context_miner.py` in `scripts/` dir in the parent folder, plus:
```bash
pip install -r requirements.txt
```

**Outputs saved by this notebook** (used later by the deployment script):
- `models/it_term_ruler/` — rule-based-only pipeline (CSV → EntityRuler). Fast, exact, zero training.
- `models/it_term_ner/` — trained statistical NER pipeline. Generalizes beyond the exact list.
- `data/it_terms.csv` — the predefined term list (copy this next to your deployment script).


In [5]:
import json
import random
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
from spacy.pipeline import EntityRuler
from spacy.tokens import DocBin
from spacy.training import Example
from spacy.scorer import Scorer
from spacy.util import filter_spans
from tqdm.auto import tqdm

PARENT_DIR = Path("..")
sys.path.insert(0, str(Path("../scripts")))

from context_miner import load_context_pipeline, build_dep_matcher, mine_candidates

random.seed(42)

DATA_DIR = Path(PARENT_DIR / "uploads" / "train")             # put raw OJT documents (.txt/.docx/.pdf) here
TERMS_CSV = Path(PARENT_DIR / "data" / "it_terms.csv")        # predefined term list
MODELS_DIR = Path(PARENT_DIR / "models")
MODELS_DIR.mkdir(exist_ok=True)

GENERIC_LABEL = "TECH_TERM"  # generic label for context-discovered terms not yet in the CSV

## Step 1 — Load the predefined IT terms (CSV)

Expected CSV format — two columns: (sample data only)

| term            | label        |
|-----------------|--------------|
| Python          | PROG_LANG    |
| React           | FRAMEWORK    |
| MySQL           | DATABASE     |
| Docker          | TOOL         |
| REST API        | CONCEPT      |

`label` is the entity category. If you don't care about categories yet, put the same
value (e.g. `IT_TERM` or `CLERICAL`) for every row — you can always split them later.

In [6]:
terms_df = pd.read_csv(TERMS_CSV)
terms_df["term"] = terms_df["term"].str.strip()
terms_df["label"] = terms_df["label"].str.strip().str.upper()
assert {"term", "label"}.issubset(terms_df.columns), "CSV must have 'term' and 'label' columns"
print(f"Loaded {len(terms_df)} predefined terms across {terms_df['label'].nunique()} categories")
terms_df.head()

Loaded 41 predefined terms across 8 categories


,term,label
0,Python,PROG_LANG
1,Java,PROG_LANG
2,JavaScript,PROG_LANG
3,TypeScript,PROG_LANG
4,C#,PROG_LANG


## Step 2 — Build the rule-based matcher (EntityRuler)

This is the exact-match backbone. It never needs "training" — it just needs the CSV.
We build it first because we'll reuse it in Step 4 to auto-annotate training data.


In [7]:
def build_ruler_pipeline(terms_df: pd.DataFrame) -> spacy.language.Language:
    nlp = spacy.blank("en")
    ruler = nlp.add_pipe("entity_ruler", config={"phrase_matcher_attr": "LOWER"})
    patterns = [
        {"label": row.label, "pattern": row.term}
        for row in terms_df.itertuples()
    ]
    ruler.add_patterns(patterns)
    return nlp

ruler_nlp = build_ruler_pipeline(terms_df)

# quick smoke test
sample = "During my OJT I used Python and Django with a MySQL database, deployed via Docker on AWS."
doc = ruler_nlp(sample)
[(ent.text, ent.label_) for ent in doc.ents]


[('Python', 'PROG_LANG'),
 ('Django', 'FRAMEWORK'),
 ('MySQL', 'DATABASE'),
 ('Docker', 'TOOL'),
 ('AWS', 'CLOUD')]

## Step 3 — Load the raw OJT documents

Point `DATA_DIR` at a folder containing the students' submitted documents. Supports
`.txt`, `.docx`, and `.pdf`. This is the *unlabeled* corpus we'll auto-annotate next.

<div class="alert alert-block alert-warning">
    <b>Note:</b> 
Since some documents were just scanned (like using CamScanner), common pdf reader wont read the documents as it is, so you need to use some kind of  Optical Character Recognition (OCR) tool to process the pixels into readable characters. This part is tricky as the computer which you will be training this one need to have these tools:

For Windows:
1. Download and run the Tesseract installer. Note its installation path (usually `C:\Program Files\Tesseract-OCR\tesseract.exe`).
2. Download poppler-windows, extract it, and add the bin folder to your system's Environment PATH variables.

For Linux and Docker Containers:
<pre>
sudo apt update
sudo apt install tesseract-ocr poppler-utils libtesseract-dev
</pre>
</div>

In [8]:
# Directory to cache extracted text on disk (one .txt per document)
TEXT_CACHE_DIR = Path(PARENT_DIR / "_text_cache")
TEXT_CACHE_DIR.mkdir(exist_ok=True)

def ocr_scanned_pdf(pdf_path) -> str:
    from pdf2image import convert_from_path, pdfinfo_from_path
    import pytesseract

    # Get page count without loading any images into memory
    info = pdfinfo_from_path(str(pdf_path))
    total_pages = info["Pages"]
    print(f"Processing {total_pages} page(s) with OCR (one at a time, 200 DPI)...")

    extracted_text = []

    # Process ONE page at a time to keep RAM low
    for page_num in range(1, total_pages + 1):
        # convert_from_path with first_page=last_page loads only 1 page
        pages = convert_from_path(
            str(pdf_path), dpi=200,
            first_page=page_num, last_page=page_num,
        )
        text = pytesseract.image_to_string(pages[0])
        extracted_text.append(text)
        del pages, text  # free the PIL image + string immediately

    result = "\n".join(extracted_text)
    del extracted_text
    return result

def extract_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == ".txt":
        return path.read_text(encoding="utf-8", errors="ignore")
    if suffix == ".docx":
        import docx
        d = docx.Document(str(path))
        return "\n".join(p.text for p in d.paragraphs)
    if suffix == ".pdf":
        return ocr_scanned_pdf(path)
    raise ValueError(f"Unsupported file type: {suffix}")

# ── Text cleaning ────────────────────────────────────────────────────
import re
import unicodedata

# Curly quotes / dashes / non-breaking spaces that Unicode normalization alone
# won't fold to plain ASCII -- map them explicitly so meaning isn't just deleted.
_PUNCT_MAP = {
    "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
    "\u2013": "-", "\u2014": "-", "\u2026": "...", "\u00a0": " ",
}

def clean_text(text: str) -> str:
    """Normalize raw extracted/OCR'd text before it's cached or used downstream:
    - Unicode-normalizes (folds ligatures/odd glyphs into standard forms)
    - de-hyphenates words split across a line break by OCR/PDF wrapping
    - maps common "smart" punctuation to plain ASCII equivalents
    - un-escapes literal backslash-n / backslash-t sequences into real whitespace
    - drops control characters and non-English/non-ASCII noise (stray symbols,
      other scripts, OCR garbage)
    - collapses runs of whitespace and blank lines

    NOTE: this keeps ASCII only. If your documents legitimately mix in Filipino/
    Taglish or other Latin-script text you want to preserve rather than strip,
    widen the final regex below from `[^\\x00-\\x7F]+` to a broader Unicode range,
    e.g. `[^\\x00-\\x7F\\u00C0-\\u024F]+`.
    """
    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text)

    for bad, good in _PUNCT_MAP.items():
        text = text.replace(bad, good)

    # de-hyphenate line-wrapped words: "inte-\ngrated" -> "integrated"
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # literal backslash-escaped newlines/tabs (not real whitespace) -> real whitespace
    text = text.replace("\\n", "\n").replace("\\t", " ")

    # drop control characters, keeping newline/tab (collapsed below)
    text = "".join(ch for ch in text if ch in "\n\t" or unicodedata.category(ch)[0] != "C")

    # drop non-English/non-ASCII noise (see docstring note above to widen this)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)

    # collapse whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    # drop lines that are just 1-2 non-word characters or isolated digits (OCR debris)
    text = "\n".join(
        ln for ln in text.split("\n")
        if len(ln.strip()) > 2 or re.match(r"[A-Za-z]{2,}", ln.strip())
    )
    return text.strip()

# quick smoke test
print(clean_text("Naging bahagi ako ng   development\u2014dagdag  \u201cexperience\u201d sa  inte-\ngration ng system.\\nSalamat!"))

# ── Discover documents and extract → clean → cache to disk ───────────
doc_paths = sorted(
    p for p in DATA_DIR.glob("**/*") if p.suffix.lower() in {".txt", ".docx", ".pdf"}
) if DATA_DIR.exists() else []
print(f"Found {len(doc_paths)} documents in {DATA_DIR}")

# ── Stream extracted text to disk instead of keeping it all in RAM ──
# Each document gets a .txt cache file; only one document's text is in
# memory at a time during extraction.
text_cache_paths: list[Path] = []
for doc_path in tqdm(doc_paths, desc="Extracting text → disk"):
    cache_file = TEXT_CACHE_DIR / f"{doc_path.stem}_{hash(str(doc_path)) & 0xFFFF:04x}.txt"
    if not cache_file.exists():          # skip if already cached from a previous run
        text = extract_text(doc_path)
        text = clean_text(text)          # ← normalize before caching
        cache_file.write_text(text, encoding="utf-8")
        del text                         # free RAM immediately
    text_cache_paths.append(cache_file)

print(f"Cached {len(text_cache_paths)} document texts in {TEXT_CACHE_DIR}/")
print("(raw_texts list is NOT held in RAM — downstream cells read from disk)")


Naging bahagi ako ng development-dagdag "experience" sa integration ng system.
Salamat!
Found 76 documents in ../uploads/train


Extracting text → disk:   0%|          | 0/76 [00:00<?, ?it/s]

Processing 1 page(s) with OCR (one at a time, 200 DPI)...


Extracting text → disk:   1%|▏         | 1/76 [00:00<00:56,  1.34it/s]

Processing 45 page(s) with OCR (one at a time, 200 DPI)...


...

Processing 39 page(s) with OCR (one at a time, 200 DPI)...


Extracting text → disk:  99%|█████████▊| 75/76 [1:03:52<00:38, 38.61s/it]

Processing 45 page(s) with OCR (one at a time, 200 DPI)...


Extracting text → disk: 100%|██████████| 76/76 [1:04:35<00:00, 51.00s/it]

Cached 76 document texts in ../_text_cache/
(raw_texts list is NOT held in RAM — downstream cells read from disk)


## Step 4 — Mine context-pattern candidates for terms *not* in the CSV

This is the piece that lets the model go beyond your predefined list. Students will
mention real tools/tech you haven't added to `it_terms.csv` yet — this step finds
them by looking at **how** the word is used, not whether it's on a list.

It looks for phrasing like:

- "I **integrated** Stripe API..."
- "...**deployed** using Kubernetes..."
- "we **migrated** to PostgreSQL"
- "**worked with** Terraform"

i.e. a trigger verb (integrate, use, deploy, migrate, configure, work with, ...)
followed by a noun phrase — and returns that noun phrase as a candidate term,
filtering out generic non-tech objects ("used my skills", "handled the project").

This needs a dependency parser (a blank pipeline has none), so it uses
`en_core_web_sm`:
```
python -m spacy download en_core_web_sm
```


In [9]:
context_nlp = load_context_pipeline("en_core_web_sm")
dep_matcher = build_dep_matcher(context_nlp)
known_terms_lower = set(terms_df["term"].str.lower())

# Process one document at a time from disk to keep RAM low
mined_per_doc = []
for cache_path in tqdm(text_cache_paths, desc="Mining context-pattern candidates"):
    text = cache_path.read_text(encoding="utf-8")
    mined_per_doc.append(mine_candidates(text, context_nlp, dep_matcher, known_terms_lower))
    del text  # free after processing

all_candidates = [c for doc_cands in mined_per_doc for c in doc_cands]
candidate_counts = Counter(c.text for c in all_candidates)
print(f"Mined {len(candidate_counts)} distinct candidate terms not in the CSV, "
      f"across {len(text_cache_paths)} documents")
pd.DataFrame(candidate_counts.most_common(30), columns=["term", "doc_count"])


Mining context-pattern candidates: 100%|██████████| 76/76 [01:35<00:00,  1.26s/it]

Mined 460 distinct candidate terms not in the CSV, across 76 documents


,term,doc_count
0,ACTIVITIES,22
1,Excel,22
2,Canva,17
3,Microsoft Excel,11
4,REFLECTIONS,5
5,Republic,4
6,Laravel and related technologies,4
7,Microsoft Word,4
8,Manolo Fortich,3
9,IV,3


**Review this list.** Two things to do with it:

1. **Right now**: anything that's clearly a real tool/tech — add it to `it_terms.csv`
   with an appropriate label. It'll be picked up as an exact match immediately, no
   retraining needed.
2. **For training**: even the ones you haven't triaged yet are used below as a
   generic `TECH_TERM` label (not a specific category) so the NER model learns the
   *pattern* — "things introduced by these verbs tend to be tech terms" — rather
   than only memorizing your exact vocabulary. That's what gives it a shot at
   recognizing genuinely new terms in future documents.


## Step 5 — Auto-annotate a training set (silver-standard labels)

Combine two label sources into one training set:
- **Exact CSV matches** (`ruler_nlp`) → their specific category label (`PROG_LANG`, `FRAMEWORK`, ...)
- **Context-pattern candidates** (Step 4) not already covered by the CSV → generic `TECH_TERM` label

Where the two overlap, the CSV's specific label wins.

> **Recommended:** review a sample of the auto-generated annotations before training
> (see Step 5b) — silver labels are only as good as your CSV + patterns, and a quick
> manual pass catches false positives and fills gaps.


In [10]:
def _clean_entity_spans(text, ents):
    """Strip leading/trailing whitespace & punctuation from entity spans.
    Drops any span that becomes empty after stripping."""
    cleaned = []
    for start, end, label in ents:
        # Strip leading whitespace/punctuation
        while start < end and (text[start].isspace() or text[start] in '.,;:!?()[]{}"\'\\'):
            start += 1
        # Strip trailing whitespace/punctuation
        while end > start and (text[end - 1].isspace() or text[end - 1] in '.,;:!?()[]{}"\'\\'):
            end -= 1
        if start < end and text[start:end].strip():
            cleaned.append((start, end, label))
    return cleaned

def make_enriched_examples(nlp_blank, text_cache_paths, mined_per_doc):
    examples = []
    for cache_path, mined in zip(text_cache_paths, mined_per_doc):
        text = cache_path.read_text(encoding="utf-8")
        if not text or not text.strip():
            continue
        doc = nlp_blank.make_doc(text)
        ruler_ents = list(ruler_nlp(text).ents)

        context_spans = []
        for cand in mined:
            offset = 0
            while True:
                idx = text.find(cand.text, offset)
                if idx == -1:
                    break
                span = doc.char_span(idx, idx + len(cand.text), label=GENERIC_LABEL, alignment_mode="contract")
                if span is not None:
                    context_spans.append(span)
                offset = idx + len(cand.text)

        # CSV (specific-label) spans win over generic context spans on overlap
        combined = sorted(list(ruler_ents) + context_spans, key=lambda s: (s.start, -(s.end - s.start)))
        combined = filter_spans(combined)

        raw_ents = [(s.start_char, s.end_char, s.label_) for s in combined]
        clean_ents = _clean_entity_spans(text, raw_ents)

        example = Example.from_dict(doc, {"entities": clean_ents})
        examples.append(example)
        del text, doc
    return examples

blank_nlp = spacy.blank("en")
all_examples = make_enriched_examples(blank_nlp, text_cache_paths, mined_per_doc)
print(f"Built {len(all_examples)} enriched training examples "
      f"(CSV terms + context-pattern candidates) from {len(text_cache_paths)} documents")


Built 75 enriched training examples (CSV terms + context-pattern candidates) from 76 documents


### Step 5b (optional but recommended) — export for manual review


In [11]:
review_path = Path(PARENT_DIR / "data" / "annotations_for_review.jsonl")
with review_path.open("w", encoding="utf-8") as f:
    for ex in all_examples:
        text = ex.reference.text
        ents = [(e.start_char, e.end_char, e.label_, e.text) for e in ex.reference.ents]
        f.write(json.dumps({"text": text, "entities": ents}) + "\n")
print(f"Wrote {review_path} — review/correct, then reload before training if desired")

# To reload corrected annotations later, rebuild Examples from the corrected spans
# rather than re-running make_enriched_examples (which would just regenerate the same silver labels).


Wrote ../data/annotations_for_review.jsonl — review/correct, then reload before training if desired


## Step 6 — Train / dev split


In [12]:
random.shuffle(all_examples)
split = int(len(all_examples) * 0.8)
train_examples, dev_examples = all_examples[:split], all_examples[split:]
print(f"Train: {len(train_examples)}  Dev: {len(dev_examples)}")

Train: 60  Dev: 15


## Step 7 — Train the statistical NER model

Starts from a blank English pipeline, adds an `ner` component, and trains it on the
auto-annotated examples (CSV labels + generic `TECH_TERM` context labels) using
spaCy's standard training loop (minibatching + dropout). Saves the best checkpoint
by dev F1.

Training on the `TECH_TERM` examples alongside the specific-category ones is what
teaches the model to flag terms it's never seen before, based on context — the
specific-category examples teach vocabulary, the `TECH_TERM` examples teach the
*pattern*.


In [13]:
def train_ner(train_examples, dev_examples, labels, n_iter=30):
    nlp = spacy.blank("en")
    ner = nlp.add_pipe("ner")
    for label in labels:
        ner.add_label(label)

    optimizer = nlp.begin_training()
    best_f1 = -1.0
    best_bytes = None

    for epoch in range(1, n_iter + 1):
        random.shuffle(train_examples)
        losses = {}
        batches = spacy.util.minibatch(train_examples, size=spacy.util.compounding(4.0, 32.0, 1.001))
        skipped = 0
        for batch in batches:
            try:
                nlp.update(batch, drop=0.2, losses=losses, sgd=optimizer)
            except ValueError:
                skipped += len(batch)
                continue

        scorer = Scorer()
        dev_scored = [nlp(ex.reference.text) for ex in dev_examples]
        scored_examples = [
            Example.from_dict(pred, {"entities": [(e.start_char, e.end_char, e.label_) for e in ref.reference.ents]})
            for pred, ref in zip(dev_scored, dev_examples)
        ]
        scores = scorer.score(scored_examples)
        f1 = scores.get("ents_f", 0.0) or 0.0

        msg = f"epoch {epoch:2d} | loss {losses.get('ner', 0.0):.2f} | "
        msg += f"P {scores.get('ents_p', 0.0):.3f} R {scores.get('ents_r', 0.0):.3f} F1 {f1:.3f}"
        if skipped:
            msg += f" (skipped {skipped} bad examples)"
        print(msg)

        if f1 > best_f1:
            best_f1 = f1
            best_bytes = nlp.to_bytes()

    if best_bytes is not None:
        nlp.from_bytes(best_bytes)
    print(f"Best dev F1: {best_f1:.3f}")
    return nlp

labels = sorted(set(terms_df["label"].unique().tolist()) | {GENERIC_LABEL})

if len(train_examples) >= 10:
    ner_nlp = train_ner(train_examples, dev_examples, labels, n_iter=30)
else:
    print("Not enough documents yet to train a reliable NER model — "
          "add more OJT documents to data/ojt_documents/ and re-run. "
          "The rule-based ruler_nlp pipeline is still fully usable on its own.")
    ner_nlp = None


epoch  1 | loss 221187.70 | P 0.000 R 0.000 F1 0.000
epoch  2 | loss 6030.25 | P 0.000 R 0.000 F1 0.000
epoch  3 | loss 3659.83 | P 0.000 R 0.000 F1 0.000
epoch  4 | loss 3236.93 | P 0.000 R 0.000 F1 0.000
epoch  5 | loss 2855.71 | P 0.349 R 0.116 F1 0.174
epoch  6 | loss 2689.68 | P 0.442 R 0.184 F1 0.260
epoch  7 | loss 2524.62 | P 0.410 R 0.130 F1 0.197
epoch  8 | loss 2421.62 | P 0.446 R 0.167 F1 0.243
epoch  9 | loss 2481.58 | P 0.422 R 0.233 F1 0.300
epoch 10 | loss 2345.52 | P 0.512 R 0.326 F1 0.398
epoch 11 | loss 2110.63 | P 0.642 R 0.210 F1 0.316
epoch 12 | loss 2222.15 | P 0.613 R 0.181 F1 0.279
epoch 13 | loss 2200.43 | P 0.559 R 0.225 F1 0.321
epoch 14 | loss 2065.31 | P 0.557 R 0.265 F1 0.359
epoch 15 | loss 2038.13 | P 0.651 R 0.213 F1 0.321
epoch 16 | loss 1895.28 | P 0.593 R 0.276 F1 0.377
epoch 17 | loss 1793.07 | P 0.538 R 0.273 F1 0.362
epoch 18 | loss 1800.92 | P 0.485 R 0.304 F1 0.374
epoch 19 | loss 1939.28 | P 0.533 R 0.340 F1 0.415
epoch 20 | loss 1912.91 | P 0

## Step 8 — Evaluate

Per-label precision / recall / F1 on the held-out dev set. Watch the `TECH_TERM`
row specifically — that's your signal for how well the model generalizes to
terms it wasn't explicitly trained on by name.


In [14]:
if ner_nlp is not None and dev_examples:
    scorer = Scorer()
    preds = [ner_nlp(ex.reference.text) for ex in dev_examples]
    scored = [
        Example.from_dict(pred, {"entities": [(e.start_char, e.end_char, e.label_) for e in ref.reference.ents]})
        for pred, ref in zip(preds, dev_examples)
    ]
    scores = scorer.score(scored)
    print("Overall:", {k: round(v, 3) for k, v in scores.items() if k.startswith("ents_") and not isinstance(v, dict)})
    print("\nPer label:")
    for label, s in (scores.get("ents_per_type") or {}).items():
        print(f"  {label:15s} P {s['p']:.3f}  R {s['r']:.3f}  F1 {s['f']:.3f}")


Overall: {'ents_p': 0.579, 'ents_r': 0.352, 'ents_f': 0.438}

Per label:
  TECH_TERM       P 0.560  R 0.330  F1 0.415
  CONCEPT         P 1.000  R 0.333  F1 0.500
  TOOL            P 0.875  R 0.636  F1 0.737
  PROG_LANG       P 0.750  R 0.600  F1 0.667
  FRAMEWORK       P 0.667  R 1.000  F1 0.800


## Step 9 — Save artifacts

Both the rule-based pipeline and the trained NER pipeline are saved. The deployment
script can use either or both (hybrid mode is recommended: exact CSV matches for
precision + NER for recall on new/unlisted terms), and separately re-runs the
context-pattern miner from Step 4 — that part needs no trained model at all, so it
keeps surfacing brand-new terms even on day one.


In [15]:
ruler_nlp.to_disk(MODELS_DIR / "it_term_ruler")
print(f"Saved rule-based pipeline to {MODELS_DIR / 'it_term_ruler'}")

if ner_nlp is not None:
    ner_nlp.to_disk(MODELS_DIR / "it_term_ner")
    print(f"Saved trained NER pipeline to {MODELS_DIR / 'it_term_ner'}")

terms_df.to_csv(MODELS_DIR / "it_terms.csv", index=False)
print("Copied terms CSV alongside the models for deployment reference")


Saved rule-based pipeline to ../models/it_term_ruler
Saved trained NER pipeline to ../models/it_term_ner
Copied terms CSV alongside the models for deployment reference


## Next steps

- **Grow the CSV**: add new terms as students' documents reveal them — the top candidates
  table from Step 4 is your shortlist. The ruler pipeline picks up new patterns instantly
  (no retraining needed for exact matches).
- **Retrain periodically**: once you've promoted some `TECH_TERM` candidates to real
  categories in the CSV (and optionally corrected annotations from Step 5b), retrain —
  those terms move from generic `TECH_TERM` to their specific category, and the model
  keeps a fresh set of `TECH_TERM` examples to keep generalizing from.
- **Tune the trigger-verb list**: `context_miner.py`'s `TRIGGER_LEMMAS` list is a starting
  point — extend it with verbs you notice your students actually use ("familiarized with",
  "utilized", "was assigned to", etc.).
- **Use the deployment script** (`deploy_term_scanner.py`) to actually scan new OJT
  submissions and store/count matches to CSV — it also re-runs the context miner to keep
  surfacing brand-new candidate terms from every new batch of submissions.
